In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
print(torch.cuda.is_available())
import torchvision
from torchvision.datasets import CIFAR10

True


In [2]:
#datasets and dataloaders
from torch.utils.data import DataLoader
import torchvision.transforms  as transforms
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
trainset = CIFAR10(root = "./data",train=True,download=True,transform=transform)
test = CIFAR10(root = "./data",train=False,download=True,transform=transform)

In [3]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [4]:
trainloader = DataLoader(trainset,batch_size = 64 , shuffle = True)
testloader = DataLoader(test,batch_size = 64 , shuffle = True)



### Build the CNN

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(2048,256),   # 128*4*4
            nn.ReLU(),
            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x
        

In [6]:
model = CNN()

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the CNN

In [ ]:
epochs = 10
for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images,labels in trainloader:
        optimizer.zero_grad()
        output = model.forward(images)
        loss = criterion(output,labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()
    print(f"epoch={epoch+1}/{epochs} & loss = {epoch_training_loss/len(trainloader)}")        

epoch=1/10 & loss = 1.3374188455474345


In [ ]:
# Evaluate the CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model(images)   # not model.forward(images)

        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

accuracy = 100 * correct_labels / total_labels
++
print(f"Accuracy = {accuracy:.2f}%")

In [ ]:
torch.save(model.state_dict(), "cifar10_cnn.pth")